Residualization  of covariates from cognitive data \
Sex, Age, Handedness, Ethnicity

In [2]:
import os

import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.impute import KNNImputer
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from sklearn.model_selection import train_test_split

from neurostatx.statistics.harmonization import neuroCombat
from neurostatx.utils.preprocessing import merge_dataframes

In [3]:
# Setting up relevant paths
repository_path = "c:/Users/Rosalie/OneDrive - Office 365/Documents/UdeS/Hiver 2026/Crédits de recherche/GitHub/FuzzyClustering-PING"
abcd_data_path = "c:/Users/Rosalie/OneDrive - Office 365/Documents/UdeS/Hiver 2026/Crédits de recherche/data/"
ping_data_dir = "c:/Users/Rosalie/OneDrive - Office 365/Documents/UdeS/Hiver 2026/Crédits de recherche/result/datagathering/"
output_dir = "c:/Users/Rosalie/OneDrive - Office 365/Documents/UdeS/Hiver 2026/Crédits de recherche/result/preprocessing/"

# Create output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [4]:
# Load PING data
ping_data = pd.read_excel(f"{ping_data_dir}/ping_data_gathered.xlsx")

In [5]:
# Separate the covariates and the variates data
covars = ping_data[["Sex", "AgeMonths", "Ethnicity", "Handedness"]].copy()
vars = ping_data[["CardSort", "Flanker", "ImitationMemory",
                  "ListSorting", "OralReading", "PatternComparison",
                    "PictureVocab"]].copy()

covars = sm.add_constant(covars)


# Instantiating an empty dataframe to store the results
residuals = pd.DataFrame(index=ping_data.index)

# Running a linear regression model for eact variate
for var in vars.columns:
    model = OLS(vars[var], covars)
    results = model.fit()
    residuals[var] = results.resid


In [6]:
# Merging the residuals with the original data
ping_data_resid = pd.concat([ping_data.drop(columns=vars), residuals], axis=1)

# Saving the residualized data.
ping_data_resid.to_excel(f"{output_dir}/ping_data_residualized.xlsx", index=False, header=True)

In [7]:
# Verifier que la residiualisation a bien fonctionné

X = sm.add_constant(ping_data[["AgeMonths", "Sex", "Ethnicity", "Handedness"]])

for col in residuals.columns:
    model = sm.OLS(residuals[col], X).fit()
    print(f"\n{col}")
    print(model.summary().tables[1])
# Coef environ 0 = pas de relation entre les covariables et les variables d'intérêt, ce qui est le but de la residualisation.
# P values élevés (1) = pas de relation significative entre les covariables et les variables d'intérêt, ce qui est également le but de la residualisation.


CardSort
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       4.746e-15      0.124   3.84e-14      1.000      -0.242       0.242
AgeMonths   4.879e-19      0.000   1.52e-15      1.000      -0.001       0.001
Sex         1.353e-16      0.036   3.71e-15      1.000      -0.072       0.072
Ethnicity   3.851e-16      0.010   3.83e-14      1.000      -0.020       0.020
Handedness   4.51e-17      0.049   9.22e-16      1.000      -0.096       0.096

Flanker
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       7.182e-16      0.121   5.92e-15      1.000      -0.238       0.238
AgeMonths   1.409e-18      0.000   4.48e-15      1.000      -0.001       0.001
Sex         2.325e-16      0.036   6.49e-15      1.000      -0.070       0.070
Ethnicity   3.678e-16      0.010 

Running an exploratory and confirmatory factorial analysis on PING cognitive data \
The CLI script will be run from the command line

In [7]:
# Using the residualized data for the further analysis.
!ExploratoryFA --in-dataset "{output_dir}/ping_data_residualized.xlsx" --out-folder "{output_dir}/PING_EFA/" \
    --id-column subjectkey --desc-columns 13 --rotation oblimin --method minres --train_dataset_size 0.5 \
    -v -s -f --random_state 1234 --nb_factors 3

2026-03-06 10:49:17 Portable-Rosalie root[56276] INFO Validating input files and creating output folder c:/Users/Rosalie/OneDrive - Office 365/Documents/UdeS/Hiver 2026/Crédits de recherche/result/preprocessing//PING_EFA/
2026-03-06 10:49:17 Portable-Rosalie root[56276] INFO Loading c:/Users/Rosalie/OneDrive - Office 365/Documents/UdeS/Hiver 2026/Crédits de recherche/result/preprocessing//ping_data_residualized.xlsx
2026-03-06 10:49:18 Portable-Rosalie root[56276] INFO Splitting into train and test datasets. Using training dataset for EFA.
2026-03-06 10:49:19 Portable-Rosalie root[56276] INFO Bartlett's test of sphericity returned a p-value of 5.388164318917512e-196 and Keiser-Meyer-Olkin (KMO)test returned a value of 0.7808097223540312.


In [8]:
# Pour que CFA trouve Graphviz (Sinon CFA fonctionne pas)
os.environ["PATH"] += r";C:\Program Files\Graphviz\bin"


In [9]:
# ConfirmatoryFA with 3 models including the cognitives test with a loading over 0.3 
!ConfirmatoryFA --in-dataset "{output_dir}/PING_EFA/test_dataset.xlsx" --out-folder "{output_dir}/PING_CFA/" \
    --id-column subjectkey --desc-columns 13 \
    --model "VA =~ OralReading + PictureVocab" \
    --model "EFPS =~ PatternComparison + CardSort + Flanker" \
    --model "MEM =~ ImitationMemory + ListSorting" \
    -v -s -f

[WinError 5] Accès refusé: 'c:/Users/Rosalie/OneDrive - Office 365/Documents/UdeS/Hiver 2026/Crédits de recherche/result/preprocessing//PING_CFA/CFA_report\\css'


2026-03-06 10:49:39 Portable-Rosalie root[40500] INFO Validating input files and creating output folder c:/Users/Rosalie/OneDrive - Office 365/Documents/UdeS/Hiver 2026/Crédits de recherche/result/preprocessing//PING_CFA/
2026-03-06 10:49:39 Portable-Rosalie root[40500] INFO Loading c:/Users/Rosalie/OneDrive - Office 365/Documents/UdeS/Hiver 2026/Crédits de recherche/result/preprocessing//PING_EFA/test_dataset.xlsx
2026-03-06 10:49:40 Portable-Rosalie root[40500] INFO Performing Confirmatory Factorial Analysis (CFA) with the following model specification:
VA =~ OralReading + PictureVocab
EFPS =~ PatternComparison + CardSort + Flanker
MEM =~ ImitationMemory + ListSorting

2026-03-06 10:49:40 Portable-Rosalie root[40500] INFO Exporting results and statistics.


In [9]:
# Applying the CFA model on the raw data
!ApplyModel --in-dataset "{output_dir}/ping_data_residualized.xlsx" --out-folder "{output_dir}/PING_CFA_Apply/" \
    --id-column subjectkey --desc-columns 13 --model "{output_dir}/PING_CFA/cfa_model.pkl" -v -s -f

2026-03-24 15:09:27 Portable-Rosalie root[2588] INFO Validating input files and creating output folder c:/Users/Rosalie/OneDrive - Office 365/Documents/UdeS/Hiver 2026/Crédits de recherche/result/preprocessing//PING_CFA_Apply/
2026-03-24 15:09:27 Portable-Rosalie root[2588] INFO Loading c:/Users/Rosalie/OneDrive - Office 365/Documents/UdeS/Hiver 2026/Crédits de recherche/result/preprocessing//ping_data_residualized.xlsx
2026-03-24 15:09:27 Portable-Rosalie root[2588] INFO Loading model
2026-03-24 15:09:33 Portable-Rosalie root[2588] INFO Applying model
2026-03-24 15:09:33 Portable-Rosalie root[2588] INFO Saving transformed dataset
